# ForkWise — Provision Infrastructure

**Fully reproducible.** Tear down and re-run anytime your lease expires.

**Prerequisites:**
1. Chameleon account with access to KVM@TACC
2. SSH keypair `forkwise-key` uploaded to Chameleon (Compute → Key Pairs)
3. `clouds.yaml` in `infra/tf/kvm/`
4. `python-blazarclient` installed (for lease creation)

**What this does:**
1. Installs Terraform + blazarclient
2. Creates a Chameleon lease (3x m1.medium, 7 days)
3. Waits for lease to become ACTIVE
4. Provisions 3 VMs via Terraform
5. Prints Part 2 commands to run on the nodes

## Step 0: Install Dependencies

In [ ]:
import subprocess, sys, os, time, json, re
from pathlib import Path
from datetime import datetime, timedelta, timezone

LOCAL_BIN = Path.home() / ".local" / "bin"
LOCAL_BIN.mkdir(parents=True, exist_ok=True)

if str(LOCAL_BIN) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{LOCAL_BIN}:{os.environ['PATH']}"


def sh(cmd, check=True, capture=True):
    print(f">>> {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if capture:
        if r.stdout.strip():
            print(r.stdout.strip()[-1000:])
        if r.stderr.strip():
            print(r.stderr.strip()[-500:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Failed: {cmd}")
    return r


def is_installed(name):
    return subprocess.run(
        f"which {name}", shell=True, capture_output=True
    ).returncode == 0


# ── Terraform ──
if not is_installed("terraform"):
    print("Installing Terraform...")
    tf_ver = "1.9.8"
    sh(f"curl -fsSL -o /tmp/tf.zip https://releases.hashicorp.com/terraform/{tf_ver}/terraform_{tf_ver}_linux_amd64.zip")
    sh(f"cd /tmp && unzip -o tf.zip && mv terraform {LOCAL_BIN}/terraform && chmod +x {LOCAL_BIN}/terraform && rm tf.zip")
else:
    print("Terraform already installed")

# ── Blazar client (for lease management) ──
try:
    import blazarclient
    print("blazarclient already installed")
except ImportError:
    print("Installing blazarclient...")
    sh(f"{sys.executable} -m pip install --user --quiet python-blazarclient")

# ── Verify ──
print()
for tool in ["terraform", "ssh"]:
    found = is_installed(tool)
    print(f"  [{'OK' if found else 'MISSING'}]  {tool}")

## Step 1: Configure Variables

In [ ]:
REPO_ROOT = Path.cwd()
TF_DIR = REPO_ROOT / "infra" / "tf" / "kvm"

# ═══ Configuration ═══
SUFFIX = "proj01"
KEYPAIR_NAME = "forkwise-key"
IMAGE_NAME = "CC-Ubuntu24.04"
LEASE_NAME = f"forkwise-k8s-{SUFFIX}"
LEASE_DAYS = 7

# VM spec for the lease reservation
VM_VCPUS = 2
VM_MEMORY_MB = 4096
VM_DISK_GB = 40
VM_COUNT = 3

# Object store credentials (fill these in)
OS_ACCESS_KEY = "YOUR_ACCESS_KEY_HERE"
OS_SECRET_KEY = "YOUR_SECRET_KEY_HERE"

# OpenStack env for CLI
os.environ["OS_CLIENT_CONFIG_FILE"] = str(TF_DIR / "clouds.yaml")
os.environ["OS_CLOUD"] = "openstack"

# Clear stale OS_ vars
for k in list(os.environ):
    if k.startswith("OS_") and k not in ("OS_CLIENT_CONFIG_FILE", "OS_CLOUD"):
        del os.environ[k]

print(f"Repo root:     {REPO_ROOT}")
print(f"Terraform dir: {TF_DIR}")
print(f"Suffix:        {SUFFIX}")
print(f"Keypair:       {KEYPAIR_NAME}")
print(f"Lease name:    {LEASE_NAME}")
print(f"VM spec:       {VM_COUNT}x ({VM_VCPUS} vCPU, {VM_MEMORY_MB}MB, {VM_DISK_GB}GB)")
print(f"Lease duration: {LEASE_DAYS} days")

## Step 2: Create Lease

Creates a Chameleon reservation for 3 VMs. If a lease with this name already
exists and is ACTIVE, it reuses it. If PENDING, it waits.

In [ ]:
# Check for existing lease with this name
r = sh(f"openstack reservation lease list --format json", check=False)
existing_lease = None
if r.returncode == 0 and r.stdout.strip():
    leases = json.loads(r.stdout)
    for lease in leases:
        if lease.get("name") == LEASE_NAME:
            existing_lease = lease
            break

if existing_lease:
    lease_id = existing_lease["id"]
    print(f"Found existing lease: {lease_id}")
    print(f"  Status will be checked in next step")
else:
    # Create new lease
    end_date = (datetime.now(timezone.utc) + timedelta(days=LEASE_DAYS)).strftime("%Y-%m-%d %H:%M")
    
    cmd = (
        f"openstack reservation lease create "
        f"--reservation resource_type=virtual:instance,"
        f"vcpus={VM_VCPUS},memory_mb={VM_MEMORY_MB},"
        f"disk_gb={VM_DISK_GB},amount={VM_COUNT} "
        f'--start-date "now" '
        f'--end-date "{end_date}" '
        f"{LEASE_NAME}"
    )
    r = sh(cmd)
    
    # Extract lease ID from output
    for line in r.stdout.split("\n"):
        if "| id" in line.lower() or ("| id " in line):
            lease_id = line.split("|")[2].strip()
            break
    else:
        # Try JSON format
        r2 = sh(f"openstack reservation lease show {LEASE_NAME} --format json", check=False)
        if r2.returncode == 0:
            lease_id = json.loads(r2.stdout)["id"]
        else:
            raise RuntimeError("Could not extract lease ID. Check output above.")
    
    print(f"Created lease: {lease_id}")

print(f"\nLease ID: {lease_id}")

## Step 3: Wait for Lease & Extract Flavor ID

The lease needs to transition from PENDING to ACTIVE before we can use
the reservation flavor. This typically takes 1-2 minutes.

In [ ]:
max_wait = 300  # 5 minutes
interval = 15
waited = 0
flavor_id = None

while waited < max_wait:
    r = sh(f"openstack reservation lease show {lease_id} --format json", check=False)
    if r.returncode != 0:
        print(f"Failed to query lease, retrying in {interval}s...")
        time.sleep(interval)
        waited += interval
        continue
    
    lease_info = json.loads(r.stdout)
    status = lease_info.get("status", "UNKNOWN")
    print(f"  [{waited}s] Lease status: {status}")
    
    if status == "ACTIVE":
        # Extract flavor_id from reservations
        reservations = lease_info.get("reservations", "")
        if isinstance(reservations, str):
            # Parse the reservation block
            for line in reservations.split("\n"):
                if "flavor_id" in line:
                    match = re.search(r'"flavor_id"\s*:\s*"([^"]+)"', line)
                    if match:
                        flavor_id = match.group(1)
                        break
        elif isinstance(reservations, list):
            for res in reservations:
                if "flavor_id" in res:
                    flavor_id = res["flavor_id"]
                    break
        elif isinstance(reservations, dict):
            flavor_id = reservations.get("flavor_id")
        
        if not flavor_id:
            # Try regex on entire output
            match = re.search(r'"flavor_id"\s*:\s*"([a-f0-9-]{36})"', r.stdout)
            if match:
                flavor_id = match.group(1)
        
        break
    elif status in ("ERROR", "TERMINATED"):
        raise RuntimeError(f"Lease is in {status} state. Delete and recreate.")
    
    time.sleep(interval)
    waited += interval

if not flavor_id:
    if status != "ACTIVE":
        raise RuntimeError(f"Lease did not become ACTIVE within {max_wait}s. Current: {status}")
    else:
        raise RuntimeError("Lease is ACTIVE but could not extract flavor_id. Run: openstack reservation lease show " + lease_id)

print(f"\nLease ACTIVE!")
print(f"Flavor ID: {flavor_id}")

## Step 4: Write terraform.tfvars & Init

In [ ]:
os.chdir(TF_DIR)

tfvars = f'''suffix       = "{SUFFIX}"
key          = "{KEYPAIR_NAME}"
image_name   = "{IMAGE_NAME}"
flavor_id    = "{flavor_id}"
'''

(TF_DIR / "terraform.tfvars").write_text(tfvars)
print("Wrote terraform.tfvars:")
print(tfvars)

# Verify clouds.yaml
if (TF_DIR / "clouds.yaml").exists():
    print("clouds.yaml: found")
else:
    raise RuntimeError("clouds.yaml not found in infra/tf/kvm/")

sh("terraform init")
sh("terraform validate")

## Step 5: Terraform Apply

Creates 3 VMs with the reserved flavor and assigns a floating IP to node1.
Takes 2-5 minutes.

In [ ]:
r = sh("terraform apply -auto-approve -parallelism=1")

ip_result = sh("terraform output -raw floating_ip", check=False)
FLOATING_IP = ip_result.stdout.strip()

if FLOATING_IP:
    print(f"\nFloating IP: {FLOATING_IP}")
else:
    raise RuntimeError("Could not get floating IP. Run: terraform output floating_ip")

## Step 6: Next Steps — Part 2 Commands

In [ ]:
from IPython.display import display, Markdown

assert FLOATING_IP, "FLOATING_IP not set - run Step 5"

instructions = f"""
---

# Part 2: Setup Cluster & Deploy

All commands below run on the VMs, not on JupyterHub.

---

## A. Upload SSH key & connect to node1

From **Windows PowerShell**:

```powershell
scp -i $HOME\.ssh\forkwise_key $HOME\.ssh\forkwise_key cc@{FLOATING_IP}:~/.ssh/id_rsa
ssh -i $HOME\.ssh\forkwise_key cc@{FLOATING_IP}
```

On node1:

```bash
chmod 600 ~/.ssh/id_rsa
```

---

## B. Pre-K8s Setup (run on ALL 3 nodes)

Run on **node1** first:

```bash
sudo apt-get update && sudo apt-get install -y curl ca-certificates jq python3 docker.io
sudo systemctl enable docker && sudo systemctl start docker
sudo usermod -aG docker $USER
sudo systemctl stop ufw 2>/dev/null; sudo systemctl disable ufw 2>/dev/null
echo "br_netfilter" | sudo tee /etc/modules-load.d/k8s.conf
sudo modprobe br_netfilter
cat <<EOF | sudo tee /etc/sysctl.d/99-kubernetes-cri.conf
net.bridge.bridge-nf-call-iptables = 1
net.bridge.bridge-nf-call-ip6tables = 1
net.ipv4.ip_forward = 1
EOF
sudo sysctl --system
```

Then do the same on **node2** and **node3**:

```bash
ssh -o StrictHostKeyChecking=no cc@192.168.1.12
# run the same block, then exit

ssh -o StrictHostKeyChecking=no cc@192.168.1.13
# run the same block, then exit
```

---

## C. Install k3s

### C1. Server on node1:

```bash
curl -sfL https://get.k3s.io | \
  INSTALL_K3S_EXEC="server --write-kubeconfig-mode 644 --node-ip 192.168.1.11 --advertise-address 192.168.1.11 --tls-san 192.168.1.11" \
  sh -
```

### C2. Get token:

```bash
K3S_TOKEN=$(sudo cat /var/lib/rancher/k3s/server/node-token)
echo $K3S_TOKEN
```

### C3. Agent on node2:

```bash
ssh -o StrictHostKeyChecking=no cc@192.168.1.12 \
  "curl -sfL https://get.k3s.io | K3S_URL=https://192.168.1.11:6443 K3S_TOKEN=$K3S_TOKEN INSTALL_K3S_EXEC='agent --node-ip 192.168.1.12' sh -"
```

### C4. Agent on node3:

```bash
ssh -o StrictHostKeyChecking=no cc@192.168.1.13 \
  "curl -sfL https://get.k3s.io | K3S_URL=https://192.168.1.11:6443 K3S_TOKEN=$K3S_TOKEN INSTALL_K3S_EXEC='agent --node-ip 192.168.1.13' sh -"
```

### C5. Verify:

```bash
sudo kubectl get nodes
# Wait until all 3 show Ready (1-2 min)
```

---

## D. Post-k3s Setup (node1)

```bash
mkdir -p ~/.kube
sudo cp /etc/rancher/k3s/k3s.yaml ~/.kube/config
sudo chown $(id -u):$(id -g) ~/.kube/config
sed -i 's|https://127.0.0.1:6443|https://192.168.1.11:6443|' ~/.kube/config

curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
kubectl patch deployment metrics-server -n kube-system --type=json \
  -p='[{{"op":"add","path":"/spec/template/spec/containers/0/args/-","value":"--kubelet-insecure-tls"}}]'

kubectl label node $(hostname) forkwise.io/entrypoint=true --overwrite
kubectl get nodes
```

---

## E. Clone Repos & Copy Manifests (node1)

```bash
cd ~
git clone https://github.com/HivanshD/ml-sys-ops-project.git
git clone https://github.com/HivanshD/mealie.git
sudo cp -r ~/ml-sys-ops-project/infra/k8s /tmp/serving-k8s
```

---

## F. Build Docker Images (node1)

### F1. Data images:

```bash
cd ~/ml-sys-ops-project/data
sudo docker build -t forkwise-ingest:local -f Dockerfile.ingest .
sudo docker build -t forkwise-feedback:local -f Dockerfile.feedback .
sudo docker build -t forkwise-batch:local -f Dockerfile.batch .
sudo docker build -t forkwise-generator:local -f Dockerfile.generator .
```

### F2. Serving:

```bash
cd ~/ml-sys-ops-project/serving
sudo docker build -t subst-serving-onnx:local -f docker/Dockerfile.fastapi_onnx .
```

### F3. Training:

```bash
cd ~/ml-sys-ops-project
sudo docker build -t forkwise-train:local -f training/docker_nvidia/Dockerfile .
```

### F4. Automation:

```bash
cd ~/ml-sys-ops-project/infra/automation
sudo docker build -t forkwise-automation:local .
```

### F5. Mealie:

```bash
cd ~/mealie
sudo docker build -t mealie:ml-ui -f docker/Dockerfile .
```

### F6. Import into k3s:

```bash
for img in forkwise-ingest:local forkwise-feedback:local forkwise-batch:local \
           forkwise-generator:local subst-serving-onnx:local forkwise-train:local \
           forkwise-automation:local mealie:ml-ui; do
  echo "Importing $img..."
  sudo docker save $img | sudo k3s ctr images import -
done
```

---

## G. Patch Manifests for Local Images (node1)

```bash
cd /tmp/serving-k8s

# Replace GHCR image references with local
sed -i 's|ghcr.io/itsnotaka/forkwise-ingest:demo|forkwise-ingest:local|g' $(grep -rl 'forkwise-ingest' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-feedback:demo|forkwise-feedback:local|g' $(grep -rl 'forkwise-feedback' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-batch:demo|forkwise-batch:local|g' $(grep -rl 'forkwise-batch' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-generator:demo|forkwise-generator:local|g' $(grep -rl 'forkwise-generator' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/mealie:ml-ui-amd64|mealie:ml-ui|g' $(grep -rl 'mealie:ml-ui' . 2>/dev/null) 2>/dev/null

# Add imagePullPolicy: Never for local images
find . -name '*.yaml' | xargs grep -l ':local' 2>/dev/null | while read f; do
  sed -i '/:local/a\        imagePullPolicy: Never' "$f"
done
find . -name '*.yaml' | xargs grep -l 'mealie:ml-ui' 2>/dev/null | while read f; do
  sed -i '/mealie:ml-ui/a\        imagePullPolicy: Never' "$f"
done
```

---

## H. Deploy Apps (node1)

```bash
kubectl apply -f /tmp/serving-k8s/apps/mealie/namespace.yaml
kubectl apply -f /tmp/serving-k8s/apps/substitution-serving/namespace.yaml

kubectl create secret generic mealie-credentials \
  --namespace forkwise-app \
  --from-literal=postgres-user=mealie \
  --from-literal=postgres-password=$(openssl rand -base64 20) \
  --from-literal=postgres-db=mealie \
  --from-literal=base-url=http://localhost:9000

kubectl apply -k /tmp/serving-k8s/apps/substitution-serving
kubectl apply -k /tmp/serving-k8s/apps/mealie

kubectl set image deployment/substitution-serving \
  substitution-serving=subst-serving-onnx:local -n forkwise-serving
kubectl set image cronjob/check-rollback \
  check-rollback=subst-serving-onnx:local -n forkwise-serving

kubectl rollout status deployment/substitution-serving -n forkwise-serving --timeout=180s
kubectl rollout status deployment/mealie-postgres -n forkwise-app --timeout=180s
kubectl rollout status deployment/mealie -n forkwise-app --timeout=240s
```

---

## I. Create Secrets (node1)

Replace the values below with your actual object-store credentials.

```bash
OS_AK="{OS_ACCESS_KEY}"
OS_SK="{OS_SECRET_KEY}"

for ns in monitoring-proj01 staging-proj01 canary-proj01 production-proj01; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic os-credentials -n $ns \
    --from-literal=OS_ENDPOINT=https://chi.tacc.chameleoncloud.org:7480 \
    --from-literal=OS_ACCESS_KEY=$OS_AK \
    --from-literal=OS_SECRET_KEY=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done

for ns in forkwise-data forkwise-serving; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic s3-credentials -n $ns \
    --from-literal=access-key=$OS_AK \
    --from-literal=secret-key=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done
```

---

## J. Deploy Rollout Stack (node1)

```bash
kubectl apply -k /tmp/serving-k8s/platform

kubectl set image deployment/automation \
  automation=forkwise-automation:local -n monitoring-proj01

kubectl rollout status deployment/prometheus -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/grafana -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/automation -n monitoring-proj01 --timeout=180s

kubectl exec -n monitoring-proj01 deploy/automation -- \
  curl -fsS -X POST http://127.0.0.1:8080/bootstrap-rollout

for env in staging canary production; do
  kubectl apply -k /tmp/serving-k8s/$env
  kubectl set image deployment/subst-serving \
    subst-serving=subst-serving-onnx:local -n ${{env}}-proj01
  kubectl rollout restart deployment/subst-serving -n ${{env}}-proj01
done
```

---

## K. Deploy Data Workloads (node1)

```bash
kubectl apply -k /tmp/serving-k8s/apps/forkwise-data

kubectl apply -f /tmp/serving-k8s/apps/forkwise-data/job-ingest.yaml
kubectl logs -n forkwise-data job/forkwise-ingest -f
kubectl wait --for=condition=complete job/forkwise-ingest -n forkwise-data --timeout=1800s

kubectl scale deployment/data-generator -n forkwise-data --replicas=1
kubectl patch cronjob batch-pipeline -n forkwise-data -p '{{"spec":{{"suspend":false}}}}'
kubectl patch cronjob drift-monitor -n forkwise-data -p '{{"spec":{{"suspend":false}}}}'
```

---

## L. Smoke Test (node1)

```bash
kubectl get nodes
kubectl get pods -A
curl -sf http://127.0.0.1:30080/health && echo "Serving OK" || echo "Serving not ready"
curl -sf http://127.0.0.1:30090 -o /dev/null && echo "Mealie OK" || echo "Mealie not ready"
```

---

## M. Access from Windows

```powershell
# Open these in separate PowerShell windows:
ssh -i $HOME\.ssh\forkwise_key -N -L 9000:127.0.0.1:30090 cc@{FLOATING_IP}
ssh -i $HOME\.ssh\forkwise_key -N -L 8000:127.0.0.1:30080 cc@{FLOATING_IP}
ssh -i $HOME\.ssh\forkwise_key -N -L 3000:127.0.0.1:30300 cc@{FLOATING_IP}
ssh -i $HOME\.ssh\forkwise_key -N -L 9090:127.0.0.1:30900 cc@{FLOATING_IP}
```

Then open:
- **Mealie:** http://localhost:9000
- **Serving:** http://localhost:8000/health
- **Grafana:** http://localhost:3000
- **Prometheus:** http://localhost:9090
"""

display(Markdown(instructions))

## Teardown

Destroy VMs and delete lease. Re-run the entire notebook to recreate.

In [ ]:
# Uncomment to tear down:
# os.chdir(TF_DIR)
# sh("terraform destroy -auto-approve")
# sh(f"openstack reservation lease delete {lease_id}", check=False)
# print("Infrastructure destroyed. Re-run notebook to recreate.")